# REE Separation for NdFeB Magnet Production

This notebook demonstrates the **difflow_ree** plugin for designing and optimizing
rare earth separation processes, specifically for producing high-purity Nd/Pr
for NdFeB permanent magnets.

## Background

NdFeB magnets are critical for:
- Electric vehicle motors
- Wind turbine generators
- Computer hard drives
- Consumer electronics

The magnetic alloy requires:
- **Nd**: Primary magnetic element (>99% purity)
- **Pr**: Often blended with Nd (didymium)
- **Dy**: Added for high-temperature performance

## Objectives

1. Model a REE separation circuit using PC88A extractant
2. Optimize for Nd/Pr recovery and purity
3. Analyze sensitivity to operating conditions
4. Estimate economics

In [ ]:
import os
os.environ['JAX_PLATFORM_NAME'] = 'cpu'

import jax
import jax.numpy as jnp
from jax import grad, jacfwd

jax.config.update("jax_enable_x64", True)

# Import difflow core
from difflow.streams import make_stream, get_flows

# Import REE plugin
from difflow_ree import (
    # Database
    get_element, list_ree_elements, get_extractant,
    # Equilibrium
    REEDistribution, get_distribution_coefficient, get_separation_factor,
    # Units
    REEExtractor, REEExtractorParams,
    # Flowsheets
    ExtractScrubStripCircuit, ExtractScrubStripParams,
    # Economics
    REEPricing, estimate_capex, estimate_opex, calculate_profit,
)

print("Available REE elements:", list_ree_elements())

## 1. REE Feed Characterization

Typical bastnasite concentrate composition after Ce removal:

In [ ]:
# Feed composition (after Ce removal)
# Flows in mol/s for a 1000 t/year plant
feed_composition = {
    "La": 0.015,   # 15%
    "Pr": 0.005,   # 5%
    "Nd": 0.020,   # 20% - main target
    "Sm": 0.002,   # 2%
    "Gd": 0.001,   # 1%
    "Dy": 0.001,   # 1% - valuable for magnets
}

print("Feed Composition (mol/s):")
print("=" * 40)
total = sum(feed_composition.values())
for elem, flow in feed_composition.items():
    props = get_element(elem)
    pct = flow / total * 100
    print(f"{elem:3s}: {flow:.4f} mol/s ({pct:5.1f}%)  - {props.group} REE")

print(f"\nTotal REE: {total:.4f} mol/s")

## 2. Extractant Selection: PC88A vs D2EHPA

PC88A is preferred for Nd/Pr separation due to higher separation factor.

In [ ]:
# Compare extractants at pH 3.5
pH = 3.5
elements = ("La", "Pr", "Nd", "Sm", "Gd", "Dy")

print(f"Distribution Coefficients at pH {pH}")
print("=" * 50)
print(f"{'Element':>8} {'D2EHPA':>12} {'PC88A':>12} {'Ratio':>10}")
print("-" * 50)

for elem in elements:
    D_d2ehpa = float(get_distribution_coefficient(elem, "D2EHPA", pH))
    D_pc88a = float(get_distribution_coefficient(elem, "PC88A", pH))
    ratio = D_pc88a / D_d2ehpa
    print(f"{elem:>8} {D_d2ehpa:>12.3f} {D_pc88a:>12.3f} {ratio:>10.2f}")

# Calculate Nd/Pr separation factor
SF_d2ehpa = float(get_separation_factor("Nd", "Pr", "D2EHPA", pH))
SF_pc88a = float(get_separation_factor("Nd", "Pr", "PC88A", pH))

print(f"\nNd/Pr Separation Factor:")
print(f"  D2EHPA: {SF_d2ehpa:.2f}")
print(f"  PC88A:  {SF_pc88a:.2f} (better for Nd/Pr separation)")

## 3. pH Optimization for Nd/Pr Separation

Find the optimal pH to maximize Nd/Pr separation.

In [ ]:
dist = REEDistribution(
    extractant="PC88A",
    elements=("La", "Pr", "Nd", "Sm", "Gd", "Dy"),
)

# Find optimal pH for Nd/Pr separation
opt_pH, max_SF = dist.optimal_pH_for_separation(
    element1="Nd",
    element2="Pr",
    pH_range=(2.0, 5.0),
)

print(f"Optimal pH for Nd/Pr separation: {opt_pH:.2f}")
print(f"Maximum separation factor: {max_SF:.2f}")

# Scan pH range
print("\npH Effect on Separation Factors:")
print(f"{'pH':>5} {'SF(Nd/Pr)':>12} {'SF(Nd/La)':>12} {'SF(Dy/Nd)':>12}")
print("-" * 45)

for pH_val in [2.0, 2.5, 3.0, 3.5, 4.0, 4.5]:
    SF_NdPr = float(dist.get_separation_factor("Nd", "Pr", pH_val))
    SF_NdLa = float(dist.get_separation_factor("Nd", "La", pH_val))
    SF_DyNd = float(dist.get_separation_factor("Dy", "Nd", pH_val))
    print(f"{pH_val:>5.1f} {SF_NdPr:>12.2f} {SF_NdLa:>12.2f} {SF_DyNd:>12.2f}")

## 4. Design 3-Section Circuit for Nd/Pr Product

Extract-Scrub-Strip configuration to produce Nd+Pr (didymium) concentrate.

In [ ]:
# Design circuit parameters
params = ExtractScrubStripParams(
    extractant="PC88A",
    elements=("La", "Pr", "Nd", "Sm", "Gd", "Dy"),
    target_elements=("Pr", "Nd"),  # Didymium product
    n_extraction_stages=12,
    n_scrubbing_stages=8,
    n_stripping_stages=5,
    extraction_pH=3.5,
    scrubbing_pH=2.5,   # Reject La
    stripping_pH=0.5,
    solvent_to_feed_ratio=1.2,
    scrub_to_solvent_ratio=0.25,
    strip_to_solvent_ratio=0.5,
)

circuit = ExtractScrubStripCircuit(params)

# Create feed stream
feed_flows = {"H2O": 50.0}  # ~1 L/s aqueous
feed_flows.update(feed_composition)

feed = make_stream(flows=feed_flows, T=298.15, P=101325.0)

# Run circuit
results = circuit(feed, T=298.15)

print("3-Section Circuit Results")
print("=" * 50)

In [ ]:
# Analyze results
print("\nTarget Element Recovery (Pr, Nd):")
for elem, rec in results["target_recovery"].items():
    print(f"  {elem}: {rec*100:.1f}%")

print(f"\nProduct Purity (Pr+Nd): {results['target_purity']*100:.1f}%")

print("\nProduct Composition:")
for elem, purity in results["product_purity"].items():
    if purity > 0.001:
        print(f"  {elem}: {purity*100:.2f}%")

print("\nImpurity Rejection:")
for elem, rej in results["impurity_rejection"].items():
    print(f"  {elem}: {rej*100:.1f}% removed")

## 5. Sensitivity Analysis with Automatic Differentiation

Use JAX gradients to analyze sensitivity to operating parameters.

In [ ]:
# Simple extractor for gradient analysis
def nd_recovery_fn(pH, n_stages, SF_ratio):
    """Calculate Nd recovery as function of operating parameters."""
    params = REEExtractorParams(
        n_stages=n_stages,
        extractant="PC88A",
        elements=("La", "Nd", "Dy"),
        pH=pH,
    )
    extractor = REEExtractor(params)
    
    feed = make_stream(
        flows={"H2O": 10.0, "La": 0.015, "Nd": 0.02, "Dy": 0.001},
        T=298.15, P=101325.0,
    )
    solvent = make_stream(
        flows={"Organic": 10.0 * SF_ratio, "La": 0.0, "Nd": 0.0, "Dy": 0.0},
        T=298.15, P=101325.0,
    )
    
    _, extract, _ = extractor(feed, solvent)
    ext_flows = get_flows(extract)
    
    return ext_flows["Nd"] / 0.02

# Base case
pH_base = 3.5
n_stages_base = 10.0
SF_base = 1.0

# Compute gradients
d_rec_d_pH = grad(nd_recovery_fn, argnums=0)(pH_base, n_stages_base, SF_base)
d_rec_d_n = grad(nd_recovery_fn, argnums=1)(pH_base, n_stages_base, SF_base)
d_rec_d_SF = grad(nd_recovery_fn, argnums=2)(pH_base, n_stages_base, SF_base)

print("Sensitivity Analysis for Nd Recovery")
print("=" * 50)
print(f"\nBase case: pH={pH_base}, N={n_stages_base}, S/F={SF_base}")
print(f"Recovery: {float(nd_recovery_fn(pH_base, n_stages_base, SF_base))*100:.1f}%")

print(f"\n∂(recovery)/∂(pH) = {float(d_rec_d_pH):.4f}")
print(f"  → +0.5 pH unit: {float(d_rec_d_pH)*0.5*100:+.2f}% change")

print(f"\n∂(recovery)/∂(n_stages) = {float(d_rec_d_n):.4f}")
print(f"  → +2 stages: {float(d_rec_d_n)*2*100:+.2f}% change")

print(f"\n∂(recovery)/∂(S/F) = {float(d_rec_d_SF):.4f}")
print(f"  → +20% solvent: {float(d_rec_d_SF)*0.2*100:+.2f}% change")

## 6. Economic Analysis

Estimate capital and operating costs, and calculate profitability.

In [ ]:
# Plant parameters
annual_capacity = 500  # tonnes REE/year

# Capital cost
capex = estimate_capex(
    annual_ree_tonnes=annual_capacity,
    n_stages_extraction=12,
    n_stages_scrubbing=8,
    n_stages_stripping=5,
    include_precipitation=True,
    year=2024,
)

print("Capital Cost Estimate")
print("=" * 40)
for item, cost in capex.items():
    print(f"{item:20s}: ${cost/1e6:,.2f} M")

print(f"\nTotal CAPEX: ${capex['total']/1e6:,.2f} M")

In [ ]:
# Operating cost
opex = estimate_opex(
    annual_ree_tonnes=annual_capacity,
    capex=capex["total"],
    extractant="PC88A",
)

print("Annual Operating Cost")
print("=" * 40)
for item, cost in opex.items():
    print(f"{item:20s}: ${cost/1e6:,.2f} M")

print(f"\nTotal OPEX: ${opex['total']/1e6:,.2f} M/year")

In [ ]:
# Revenue calculation
pricing = REEPricing()

# Assume 80% of feed is Nd+Pr, 90% recovery, 95% purity
nd_production_kg = annual_capacity * 1000 * 0.5 * 0.9  # 50% Nd in product
pr_production_kg = annual_capacity * 1000 * 0.15 * 0.9  # 15% Pr
dy_production_kg = annual_capacity * 1000 * 0.02 * 0.95  # 2% Dy, separate product

nd_revenue = nd_production_kg * pricing.get_price("Nd", "99.9%", "oxide")
pr_revenue = pr_production_kg * pricing.get_price("Pr", "99.9%", "oxide")
dy_revenue = dy_production_kg * pricing.get_price("Dy", "99.9%", "oxide")

total_revenue = nd_revenue + pr_revenue + dy_revenue

print("Revenue Estimate")
print("=" * 40)
print(f"Nd ({nd_production_kg/1000:.0f} t/y): ${nd_revenue/1e6:,.2f} M")
print(f"Pr ({pr_production_kg/1000:.0f} t/y): ${pr_revenue/1e6:,.2f} M")
print(f"Dy ({dy_production_kg/1000:.0f} t/y): ${dy_revenue/1e6:,.2f} M")
print(f"\nTotal Revenue: ${total_revenue/1e6:,.2f} M/year")

In [ ]:
# Profitability
profit = calculate_profit(
    revenue=total_revenue,
    opex=opex["total"],
    capex=capex["total"],
)

print("Profitability Analysis")
print("=" * 40)
print(f"Revenue:      ${profit['revenue']/1e6:>8,.2f} M/year")
print(f"OPEX:         ${profit['opex']/1e6:>8,.2f} M/year")
print(f"EBITDA:       ${profit['ebitda']/1e6:>8,.2f} M/year")
print(f"Depreciation: ${profit['depreciation']/1e6:>8,.2f} M/year")
print(f"Net Income:   ${profit['net_income']/1e6:>8,.2f} M/year")
print(f"\nPayback Period: {profit['payback_years']:.1f} years")
print(f"ROI: {profit['roi']*100:.1f}%")

## Summary

This notebook demonstrated the **difflow_ree** plugin for:

1. **Database access** - REE properties, extractant data, separation factors
2. **Equilibrium modeling** - pH-dependent distribution coefficients
3. **Process design** - 3-section extract-scrub-strip circuit
4. **Sensitivity analysis** - Automatic differentiation for gradients
5. **Economic analysis** - CAPEX, OPEX, and profitability metrics

The differentiable framework enables:
- Rapid optimization of operating conditions
- Sensitivity analysis for process design
- Integration with gradient-based optimizers

## Next Steps

- Try different extractants (D2EHPA for heavy REE)
- Optimize number of stages and flow ratios
- Model complete separation train with Ce removal
- Perform uncertainty analysis on REE prices